# Train CVAE for generative candle quality (steven3)

Run this notebook's kernel connected to a Colab runtime (VS Code: kernel picker top-right -> "Select Another Kernel" -> Google Colab -> pick a GPU runtime).

This is a **separate, focused notebook** from `colab_train.ipynb` -- it does not train PatchTST or run the trading backtest. It trains and evaluates the two CVAE checkpoints from the generative-quality pivot (see `cvae_direction_collapse.md`'s "generative pivot" discussion): a decoder that predicts a per-component variance and trains against a real NLL, compared against an architecture-matched plain-MSE baseline. `colab_train.ipynb` still trains/evaluates PatchTST + the trading-framed CVAE and stays untouched.

Clones/checks out the `steven3` branch specifically. Run the cells top to bottom.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
import os

REPO_URL = "https://github.com/WoodyChang21/ECE1508_GenAI.git"
BRANCH = "steven3"

if not os.path.isdir("ECE1508_GenAI"):
    !git clone -b {BRANCH} {REPO_URL}
else:
    # fetch + hard reset, not a plain pull -- see colab_train.ipynb's clone cell for why
    # (a pull can fail outright on local changes and Colab just prints the error and
    # moves on, leaving training to silently proceed on stale code).
    !cd ECE1508_GenAI && git fetch origin {BRANCH} && git reset --hard origin/{BRANCH}

%cd ECE1508_GenAI
!git log --oneline -1

In [ ]:
# torch is preinstalled on Colab; just need mplfinance + pyyaml
!pip install -q mplfinance pyyaml

In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

## Sanity checks (data pipeline + generative-metrics tests)

Cheap to run first -- confirms the feature/window/loss/metrics logic before committing to a long training run.

In [ ]:
!pip install -q pytest
!python -m pytest steven/tests/ -v

## Momentum feature setup (EMA9/EMA21 + RSI-14 + VIX)

Both configs below train on the base hourly OHLCV plus three added features (`src/momentum_pipeline.py`) -- EMA9/EMA21 crossover, RSI-14, and VIX (previous trading day's close). VIX isn't in the hourly parquet, so it's pulled fresh here via `yfinance` before training. Fresh pull each run rather than committing the parquet -- cheap, avoids the data going stale, matches `colab_train.ipynb`'s own convention.

In [ ]:
!pip install -q -r steven/requirements-probe.txt
!python steven/src/collect_vix_yfinance.py

## Train CVAE -- NLL + learned variance (configs/cvae_generative.yaml)

Same architecture as `configs/cvae_hourly_momentum.yaml` (momentum features on, `z_dim=8`/`ctx_dropout=0.3`/`decoder_ctx_dim=8`), but the decoder predicts a per-component variance and trains against a real NLL (Laplace on open_ret/body_ret, Gaussian on wicks/volume) instead of plain MSE, with a 5-epoch mean-only warmup. `w_direction` is disabled (0.0). Writes `steven/outputs/cvae_checkpoint_generative.pt`.

In [ ]:
!python steven/src/train_cvae.py --config steven/configs/cvae_generative.yaml --device auto

## Train CVAE -- MSE baseline, same architecture (configs/cvae_generative_mse.yaml)

Identical architecture/momentum-features/`w_direction=0` recipe as the NLL run above, but `reconstruction: mse` -- isolates the loss as the single changed variable for a fair before/after comparison (the pre-upgrade `cvae_checkpoint.pt` can't be reused for this: the decoder's output width changed 15->30, so it no longer loads into the current model class at all). Writes `steven/outputs/cvae_checkpoint_generative_mse_baseline.pt`.

In [ ]:
!python steven/src/train_cvae.py --config steven/configs/cvae_generative_mse.yaml --device auto

## Evaluate generative quality: diversity, calibration, context-sensitivity

Runs `src/evaluate_generative.py` over a full deterministic rolling-window pass (`ctx_bars=70`, `k=32` samples/window) for each checkpoint, reporting per-component/bar diversity, CRPS (+ skill score vs. a context-blind climatology baseline), rank-histogram extreme-rank fraction, PIT/coverage (NLL checkpoint only), and the context-sensitivity `effect_ratio` (does generated output shift across realized-volatility regimes the way real data does?). Writes a metrics JSON + a regime x k-samples grid and a diversity fan chart per checkpoint.

In [ ]:
!python steven/src/evaluate_generative.py \
  --cvae-checkpoint steven/outputs/cvae_checkpoint_generative.pt \
  --metrics-out steven/outputs/generative_metrics_nll.json \
  --plots-dir steven/outputs/generative_plots_nll \
  --device auto

In [ ]:
!python steven/src/evaluate_generative.py \
  --cvae-checkpoint steven/outputs/cvae_checkpoint_generative_mse_baseline.pt \
  --metrics-out steven/outputs/generative_metrics_mse_baseline.json \
  --plots-dir steven/outputs/generative_plots_mse_baseline \
  --device auto

## Quick side-by-side comparison

Headline numbers only -- open the two plot directories above for the full visual picture (regime grid + diversity fan).

In [ ]:
import json

nll = json.load(open("steven/outputs/generative_metrics_nll.json"))
mse = json.load(open("steven/outputs/generative_metrics_mse_baseline.json"))

print(f"{'metric (bar0_body_ret)':40s} {'mse':>12s} {'nll':>12s}")
for k in ["variance_ratio", "crps", "extreme_rank_fraction", "diversity_std"]:
    m, n = mse["per_component"]["bar0_body_ret"][k], nll["per_component"]["bar0_body_ret"][k]
    print(f"{k:40s} {m:12.4f} {n:12.4f}")
print(f"{'crps_skill_score_bar0_body_ret':40s} {mse['crps_skill_score_bar0_body_ret']:12.4f} "
      f"{nll['crps_skill_score_bar0_body_ret']:12.4f}")
for comp in ["open_ret", "body_ret", "upper_wick", "lower_wick"]:
    key = f"bar0_{comp}"
    m = mse["context_sensitivity_realized_vol_tercile"][key]["effect_ratio_mean"]
    n = nll["context_sensitivity_realized_vol_tercile"][key]["effect_ratio_mean"]
    print(f"{'effect_ratio_mean ' + key:40s} {m:12.4f} {n:12.4f}")

## Sync results back to GitHub

Commits `steven/outputs/` (checkpoints, metrics JSONs, plots) from this Colab runtime and pushes to the `steven3` branch -- same pattern as `colab_train.ipynb`'s sync cell. No `v1.md` update here -- that report is downstream of the trading-framed workflow and isn't touched by this generative-quality pass.

Needs a GitHub personal access token with `repo` write scope for this push only -- entered via `getpass` below, never written to the notebook or committed anywhere.

In [ ]:
# %%bash
# git fetch origin steven3
# git merge origin/steven3 --no-edit

In [ ]:
import getpass

token = getpass.getpass("GitHub PAT (repo write, used only for this push): ")

In [ ]:
%%bash -s "$token"
TOKEN="$1"
if [ -z "$TOKEN" ]; then
  echo "Token was empty -- re-run the getpass cell above and actually paste your PAT before pressing Enter." >&2
  exit 1
fi
git config user.email "colab@ephemeral.local"
git config user.name "Colab Runtime"
git add steven/outputs
if git diff --cached --quiet; then
  echo "Nothing new to commit -- outputs unchanged from last commit."
else
  git commit -m "Retrain generative-quality CVAE (NLL + MSE baseline) from Colab run"
fi
git push "https://${TOKEN}@github.com/WoodyChang21/ECE1508_GenAI.git" HEAD:steven3

### Fallback: zip + browser download

Use this if the push cell above fails (e.g. bad token) -- zips `steven/outputs` for a manual download instead.

In [ ]:
!zip -r outputs.zip steven/outputs